In [ ]:
from core.datasets import MedicalDataset
from core.evaluation import evaluate_model
from core.preprocessing import train_transform
from core.visualization import draw_img
from core.training import train_model
from core.helpers import repeat_samples, to_tensor
from core.models import AneurysmClassifierResNet, AneurysmClassifierMobileNet, AneurysmClassifierShuffleNet
from scipy.ndimage import center_of_mass
import torch
import os
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader
from torch.optim import AdamW, Adam

In [ ]:
def scan_masks_sizes(segmentations):
    x_diff, y_diff = [], []
    max_x_dist = max_x_dist_idx = max_y_dist = max_y_dist_idx = 0
    curr_idx = -1

    for batch in range(segmentations.shape[0]):
        scanned = _scan_mask(segmentations[batch, :, :, :])
        curr_idx += 1
        
        if not scanned:
            continue

        x_diff.append(scanned[1] - scanned[0])
        y_diff.append(scanned[3] - scanned[2])

        if max_x_dist < x_diff[-1]:
            max_x_dist = x_diff[-1]
            max_x_dist_idx = curr_idx

        if max_y_dist < y_diff[-1]:
            max_y_dist = y_diff[-1]
            max_y_dist_idx = curr_idx

    return np.array(x_diff), np.array(y_diff), max_x_dist_idx, max_y_dist_idx

def delete_images_without_cow(x, y):
    keep = np.any(x[:, :, :, 5:] != 0, axis=(1, 2, 3))
    return x[keep], y[keep]

def _scan_mask(seg):
    mask = np.where(seg > 0)
    min_x = mask[1].min()
    max_x = mask[1].max()
    min_y = mask[0].min()
    max_y = mask[0].max()
    return min_x, max_x, min_y, max_y

def get_centroids(segmentations):
    labels = np.ones_like(segmentations).cumsum(axis=0)
    indices = np.arange(1, segmentations.shape[0] + 1)
    return center_of_mass(segmentations > 0, labels, indices)

def center_crop_to_fixed_size(data, HEIGHT_HALF, WIDTH_HALF):
    cropped_data = []
    centroids = get_centroids(data[:, :, :, 5:])

    for i, centroid in enumerate(centroids):
        y = round(centroid[1])
        x = round(centroid[2])

        max_y = y + HEIGHT_HALF
        max_x = x + WIDTH_HALF
        min_y = y - HEIGHT_HALF
        min_x = x - WIDTH_HALF

        after_x = max(0, max_x - data.shape[2])
        after_y = max(0, max_y - data.shape[1])
        before_x = max(0, -min_x)
        before_y = max(0, -min_y)

        sliced_arr = data[i, min_y + before_y : max_y, min_x + before_x : max_x, :]

        cropped_data.append(
            np.pad(
            sliced_arr,
            ((before_y, after_y), (before_x, after_x), (0, 0)),
            mode='constant'
            )
        )

    return cropped_data

In [ ]:
train_folder = Path('C:/Users/dawid/OneDrive/Pulpit/aneurysm classifier/data/train')
test_folder = Path('C:/Users/dawid/OneDrive/Pulpit/aneurysm classifier/data/test')

In [ ]:
x_train = np.load(os.path.join(train_folder, 'train_images.npy'))
y_train = np.load(os.path.join(train_folder, 'train_labels.npy'))
x_test = np.load(os.path.join(test_folder, 'test_images.npy'))
y_test = np.load(os.path.join(test_folder, 'test_labels.npy'))

In [ ]:
x_train, y_train = delete_images_without_cow(x_train, y_train)

In [ ]:
ret = _scan_mask(x_test[0, :, :, 5:])
print(ret)

fig, axes = plt.subplots(1, 5, figsize=(15, 10))

for i in range(5):
    axes[i].imshow(x_test[0, :, :, i + 5])

plt.show()

In [ ]:
x_diff, y_diff, x_idx, y_idx = scan_masks_sizes(x_train[:, :, :, 5:])
xp_cut = x_diff[x_diff < np.percentile(x_diff, 98)].max()
yp_cut = y_diff[y_diff < np.percentile(y_diff, 98)].max()

print(f'Max\nX: {x_diff.max()}, Y: {y_diff.max()}')
print(f'Percetiles\nX: {xp_cut}, Y: {yp_cut}')

In [ ]:
draw_img(x_train[x_idx])
ret = _scan_mask(x_train[x_idx, :, :, 5:])
print(ret[1] - ret[0], ret[3] - ret[2])
print(ret)

In [ ]:
draw_img(x_train[y_idx])
ret = _scan_mask(x_train[y_idx, :, :, 5:])
print(ret[1] - ret[0], ret[3] - ret[2])
print(ret)

In [ ]:
WIDTH = 96
HEIGHT = 64
WIDTH_HALF = WIDTH // 2
HEIGHT_HALF = HEIGHT // 2

In [ ]:
example = center_crop_to_fixed_size(x_train[0:2], HEIGHT_HALF, WIDTH_HALF)
draw_img(example[1])

In [ ]:
draw_img(x_train[1])

In [ ]:
x_train, y_train = repeat_samples(x_train, y_train, 20)
x_train_tensor, y_train_tensor, x_test_tensor, y_test_tensor = to_tensor(x_train, y_train, x_test, y_test)

In [ ]:
train_dataset = MedicalDataset(x_train_tensor, y_train_tensor, train_transform)
test_dataset = MedicalDataset(x_test_tensor, y_test_tensor)

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
lr = 1e-4
num_epochs = 8

model = AneurysmClassifierResNet(10, 14)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = Adam(lr=lr, params=params, weight_decay=1e-4)

class_weights = torch.tensor(
    [y_train.shape[0] / counts for counts in torch.bincount(y_train_tensor[:, 0])], dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
best_model_weights, history = train_model(
    model, optimizer, criterion, train_loader, test_loader, num_epochs, device)

In [ ]:
model.load_state_dict(best_model_weights)

In [ ]:
evaluate_model(model, device, test_loader)

In [ ]:
lr = 1e-3
num_epochs = 8

mobilenet_model = AneurysmClassifierMobileNet(10, 14)
mobilenet_model.to(device)

mn_optimizer = Adam(mobilenet_model.parameters(), lr=lr)

class_weights = torch.tensor(
    [y_train.shape[0] / counts for counts in torch.bincount(y_train_tensor[:, 0])], dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
best_model_weights, history = train_model(
    mobilenet_model, mn_optimizer, criterion, train_loader, test_loader, num_epochs, device)

In [ ]:
mobilenet_model.load_state_dict(best_model_weights)

In [ ]:
evaluate_model(mobilenet_model, device, test_loader)

In [ ]:
lr = 1e-4
num_epochs = 8

shuffle_model = AneurysmClassifierShuffleNet(10, 14)
shuffle_model.to(device)

params = [p for p in shuffle_model.parameters() if p.requires_grad]
optimizer = AdamW(lr=lr, params=params, weight_decay=1e-5)

class_weights = torch.tensor(
    [y_train.shape[0] / counts for counts in torch.bincount(y_train_tensor[:, 0])], dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
best_model_weights, history = train_model(shuffle_model, optimizer, criterion, train_loader, test_loader, num_epochs, device)

In [ ]:
evaluate_model(shuffle_model, device, test_loader)